In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context("notebook")
# 1. LOAD DATA
df_internal = pd.read_csv('data_internal.csv')
df_external = pd.read_csv('data_external.csv')

# 2. LABEL SOURCES
df_internal['Source'] = 'ViennaAIdb'
df_external['Source'] = 'MIMIC'

combined_df = pd.concat([df_internal, df_external], axis=0)
numerical_cols = combined_df.select_dtypes(include=['number']).columns.tolist()
numerical_cols.remove('encounterId')
# 3. SETUP PLOT GRID


In [ ]:
rename_mapping = {
    'utcChartTime': 'Observation Timestamp',
    'encounterId': 'Encounter ID',
    'age': 'Patient Age (years)',
    'sex_or_gender': 'Biological Sex',
    'blood_pressure_diastolic_mmHg': 'Diastolic Blood Pressure (mmHg)',
    'blood_pressure_mean_mmHg': 'Mean Arterial Pressure (mmHg)',
    'blood_pressure_systolic_mmHg': 'Systolic Blood Pressure (mmHg)',
    'combined_vaso': 'Vasoactive-Inotropic Score',
    'colloids_ml': 'Colloid Fluid Volume (mL)',
    'fluids_ml': 'Crystalloid Fluid Volume (mL)',
    'fibrinogen_mg/dl': 'Plasma Fibrinogen (mg/dL)',
    'lactate_mmol/l': 'Serum Lactate (mmol/L)',
    'platelet_count_G/l': 'Platelet Count (x10^9/L)',
    'hemoglobin_g/dl': 'Hemoglobin (g/dL)',
    'base_excess_mmol/l': 'Base Excess (mmol/L)',
    'harnk_ml': 'Urine Output (mL)',
    'drain_sum': 'Surgical Drain Output (mL)',
    'heart_rate': 'Heart Rate (bpm)',
    'respiratory_rate': 'Respiratory Rate (breaths/min)',
    'spo2': 'SpO2 (%)',
    'blood_input': 'Transfused Blood Products (mL)'
}

# Drop the duplicate column if it exists in your combined_df
if 'sex_or_gender.1' in combined_df.columns:
    combined_df = combined_df.drop(columns=['sex_or_gender.1'])

# Apply the mapping to rename the columns
combined_df = combined_df.rename(columns=rename_mapping)

# 2. DEFINE NUMERICAL COLUMNS FOR PLOTTING
# We want to plot all columns EXCEPT identifiers, timestamps, categories, and the target/source.
exclude_cols = ['Observation Timestamp', 'Encounter ID', 'Biological Sex', 'Source']
numerical_cols = [col for col in combined_df.columns if col not in exclude_cols]

# 3. SET PUBLICATION-READY STYLE
sns.set_theme(style="whitegrid", context="paper") # 'paper' scales fonts nicely for manuscripts

In [ ]:
n_vars = len(numerical_cols)
fig, axes = plt.subplots(nrows=n_vars, ncols=2, figsize=(15, 4 * n_vars))

if n_vars == 1:
    axes = axes.reshape(1, -1)

print(f"Generating balanced plots (max 5,000 samples each) for {n_vars} variables...")



# 4. LOOP & BALANCE DATA PER COLUMN
for i, col in enumerate(numerical_cols):
    
    # --- A. Extract valid values (drop NaNs) ---
    vienna_vals = combined_df[(combined_df['Source'] == 'ViennaAIdb')][col].dropna()
    mimic_vals = combined_df[(combined_df['Source'] == 'MIMIC')][col].dropna()
    
    # --- B. Determine Sample Size (Cap at 5000) ---
    # We take the minimum of:
    # 1. 5000 (your requested max)
    # 2. Vienna count
    # 3. MIMIC count
    # This guarantees we never try to sample more data than exists.
    n_samples = min(5000, len(vienna_vals), len(mimic_vals))

    # Skip plot if data is effectively empty
    if n_samples == 0:
        axes[i, 0].text(0.5, 0.5, "Insufficient Data", ha='center', va='center')
        axes[i, 1].text(0.5, 0.5, "Insufficient Data", ha='center', va='center')
        continue

    # --- C. Sample both to the same size ---
    v_sampled = vienna_vals.sample(n=n_samples, random_state=42)
    m_sampled = mimic_vals.sample(n=n_samples, random_state=42)
    
    # Create temp DataFrame for plotting
    plot_data = pd.DataFrame({
        'Value': pd.concat([v_sampled, m_sampled]),
        'Source': ['ViennaAIdb'] * n_samples + ['MIMIC'] * n_samples
    })

    # --- D. PLOT ---
    # Left: Histogram
    sns.histplot(
        data=plot_data, x='Value', hue='Source', 
        kde=True, element="step", bins=30,
        palette={'ViennaAIdb': "#005B96", 'MIMIC': "#00A6A6"},
        alpha=0.3, ax=axes[i, 0],
    )
    leg = axes[i, 0].get_legend()
    if leg is not None:
        leg.set_title(None)
    axes[i, 0].set_title(f'{col}: Distribution (N={n_samples} each)', fontsize=12)
    axes[i, 0].set_ylabel('Frequency')
    axes[i, 0].set_xlabel(col)
    
    # Right: Boxplot
    sns.boxplot(
        data=plot_data, x='Value', y='Source', 
        palette={'ViennaAIdb': "#005B96", 'MIMIC':  "#00A6A6"},
        ax=axes[i, 1]
    )
    axes[i, 1].set_title(f'{col}: Boxplot', fontsize=12)
    axes[i, 1].set_ylabel('')
    axes[i, 1].set_xlabel(col)

# 5. SAVE
plt.tight_layout()
plt.savefig('balanced_5k_comparison.png', dpi=300)
print(f"Comparison image saved as 'balanced_5k_comparison.png'.")
plt.show()

In [ ]:
numerical_cols